In [18]:
import pandas as pd
import numpy as np
import numpy.random as rand
import scipy.stats
from gtfparse import read_gtf
import polars as pl
import matplotlib as mpl
import matplotlib.pyplot as plt

In [3]:
# Read in expression table
rna = pd.read_table('/home/eholc/HW8/GD660.TrQuantRPKM.txt')
rna

,TargetID,Gene_Symbol,Chr,Coord,HG00096.1.M_111124_6,HG00097.7.M_120219_2,HG00099.1.M_120209_6,HG00099.5.M_120131_3,HG00100.2.M_111215_8,HG00101.1.M_111124_4,...,NA20810.2.M_111215_7,NA20811.1.M_111124_5,NA20812.2.M_111216_6,NA20813.5.M_120131_1,NA20814.2.M_111215_6,NA20815.5.M_120131_5,NA20816.3.M_120202_7,NA20819.3.M_120202_2,NA20826.1.M_111124_1,NA20828.2.M_111216_8
0,ENST00000454411.1,ENSG00000237851.1,6,143109260,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
1,ENST00000425977.1,ENSG00000225538.1,11,55850277,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.03876,0.00000
2,ENST00000417072.1,ENSG00000212855.5,Y,9578193,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3,ENST00000552753.1,ENSG00000257527.1,16,18505708,0.70561,0.66697,0.64004,0.26195,0.34695,1.49208,...,0.87085,0.94950,0.95837,0.51002,0.29422,0.22960,0.58671,0.27674,0.53630,0.17139
4,ENST00000470114.1,ENSG00000243765.1,15,58442766,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183081,ENST00000569773.1,ENSG00000006007.7,16,19533467,0.00000,0.25167,0.00000,0.29288,0.00000,0.00000,...,0.00000,0.00000,19.04685,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
183082,ENST00000390751.1,ENSG00000212040.1,14,101498324,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
183083,ENST00000559497.1,ENSG00000259738.1,15,59157205,0.00000,0.13191,0.00000,0.00000,0.15789,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
183084,ENST00000416709.1,ENSG00000230711.2,6,168690030,0.20363,0.18754,0.18268,0.27913,0.28883,0.20242,...,0.26582,0.19322,0.28367,0.30192,0.20901,0.15291,0.27677,0.59387,0.22323,0.30439


In [4]:
# 1. Determine which transcript has the largest range in RPKM values across all samples.
ranges = {}                                 # Create dictionary to store ranges for each row
rpkm_columns = rna.iloc[:, 4:]              # Subset only the columns with expression data

for index, row in rpkm_columns.iterrows():  # Loop through each row
    max_value = row.max()                   # Find the max of each row
    min_value = row.min()                   # Find the min of each row
    range_value = max_value - min_value     # Calculate range of each row
    
    ranges[index] = range_value             # Add range value to row index
    
max_range = max(ranges, key=ranges.get)     # Find index with the max range value
my_gene = rna.loc[max_range]                # Find gene with the max range index

In [5]:
rpkm_columns

,HG00096.1.M_111124_6,HG00097.7.M_120219_2,HG00099.1.M_120209_6,HG00099.5.M_120131_3,HG00100.2.M_111215_8,HG00101.1.M_111124_4,HG00102.3.M_120202_8,HG00103.4.M_120208_3,HG00104.1.M_111124_5,HG00105.1.M_120209_7,...,NA20810.2.M_111215_7,NA20811.1.M_111124_5,NA20812.2.M_111216_6,NA20813.5.M_120131_1,NA20814.2.M_111215_6,NA20815.5.M_120131_5,NA20816.3.M_120202_7,NA20819.3.M_120202_2,NA20826.1.M_111124_1,NA20828.2.M_111216_8
0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
1,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.03876,0.00000
2,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3,0.70561,0.66697,0.64004,0.26195,0.34695,1.49208,0.82663,0.26740,0.15260,0.94254,...,0.87085,0.94950,0.95837,0.51002,0.29422,0.22960,0.58671,0.27674,0.53630,0.17139
4,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183081,0.00000,0.25167,0.00000,0.29288,0.00000,0.00000,0.08559,0.34944,0.00000,0.00000,...,0.00000,0.00000,19.04685,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
183082,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
183083,0.00000,0.13191,0.00000,0.00000,0.15789,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
183084,0.20363,0.18754,0.18268,0.27913,0.28883,0.20242,0.46896,0.50457,0.16939,0.26630,...,0.26582,0.19322,0.28367,0.30192,0.20901,0.15291,0.27677,0.59387,0.22323,0.30439


In [6]:
print(my_gene)

TargetID                ENST00000419932.1
Gene_Symbol             ENSG00000226958.1
Chr                                     X
Coord                           108297792
HG00096.1.M_111124_6           2338.87768
                              ...        
NA20815.5.M_120131_5           2809.14257
NA20816.3.M_120202_7           1655.52294
NA20819.3.M_120202_2            830.08642
NA20826.1.M_111124_1           3222.13354
NA20828.2.M_111216_8             820.5487
Name: 4555, Length: 664, dtype: object


In [7]:
print(my_gene["Gene_Symbol"])

ENSG00000226958.1


In [8]:
print(max_range)

4555


In [9]:
# Using the file gencode.v12.annotation.gtf, determine the name of the corresponding gene.
# Read in annotation file
anno = read_gtf('/home/eholc/HW8/gencode.v12.annotation.gtf')

# Change anno file from Polars DataFrame to a pandas DataFrame
anno = anno.to_pandas()
anno

INFO:root:Extracted GTF attributes: ['gene_id', 'transcript_id', 'gene_type', 'gene_status', 'gene_name', 'transcript_type', 'transcript_status', 'transcript_name', 'level', 'havana_gene', 'havana_transcript', 'ont', 'tag', 'ccdsid']


,seqname,source,feature,start,end,score,strand,frame,gene_id,transcript_id,...,gene_name,transcript_type,transcript_status,transcript_name,level,havana_gene,havana_transcript,ont,tag,ccdsid
0,chr1,HAVANA,gene,11869,14412,NaN,+,0,ENSG00000223972.4,ENSG00000223972.4,...,DDX11L1,pseudogene,KNOWN,DDX11L1,2,OTTHUMG00000000961.2,,,,
1,chr1,HAVANA,transcript,11869,14409,NaN,+,0,ENSG00000223972.4,ENST00000456328.2,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
2,chr1,HAVANA,exon,11869,12227,NaN,+,0,ENSG00000223972.4,ENST00000456328.2,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
3,chr1,HAVANA,exon,12613,12721,NaN,+,0,ENSG00000223972.4,ENST00000456328.2,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
4,chr1,HAVANA,exon,13221,14409,NaN,+,0,ENSG00000223972.4,ENST00000456328.2,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2616292,chrM,ENSEMBL,transcript,15888,15953,NaN,+,0,ENSG00000210195.2,ENST00000387460.2,...,J01415.22,Mt_tRNA,NOVEL,J01415.22-201,3,,,,,
2616293,chrM,ENSEMBL,exon,15888,15953,NaN,+,0,ENSG00000210195.2,ENST00000387460.2,...,J01415.22,Mt_tRNA,NOVEL,J01415.22-201,3,,,,,
2616294,chrM,ENSEMBL,gene,15956,16023,NaN,-,0,ENSG00000210196.2,ENSG00000210196.2,...,J01415.23,Mt_tRNA,NOVEL,J01415.23,3,,,,,
2616295,chrM,ENSEMBL,transcript,15956,16023,NaN,-,0,ENSG00000210196.2,ENST00000387461.2,...,J01415.23,Mt_tRNA,NOVEL,J01415.23-201,3,,,,,


In [10]:
# Index anno dataframe by gene_id
anno = anno.set_index('gene_id')
anno

,seqname,source,feature,start,end,score,strand,frame,transcript_id,gene_type,...,gene_name,transcript_type,transcript_status,transcript_name,level,havana_gene,havana_transcript,ont,tag,ccdsid
gene_id,,,,,,,,,,,,,,,,,,,,,
ENSG00000223972.4,chr1,HAVANA,gene,11869,14412,NaN,+,0,ENSG00000223972.4,pseudogene,...,DDX11L1,pseudogene,KNOWN,DDX11L1,2,OTTHUMG00000000961.2,,,,
ENSG00000223972.4,chr1,HAVANA,transcript,11869,14409,NaN,+,0,ENST00000456328.2,pseudogene,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
ENSG00000223972.4,chr1,HAVANA,exon,11869,12227,NaN,+,0,ENST00000456328.2,pseudogene,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
ENSG00000223972.4,chr1,HAVANA,exon,12613,12721,NaN,+,0,ENST00000456328.2,pseudogene,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
ENSG00000223972.4,chr1,HAVANA,exon,13221,14409,NaN,+,0,ENST00000456328.2,pseudogene,...,DDX11L1,processed_transcript,KNOWN,DDX11L1-002,2,OTTHUMG00000000961.2,OTTHUMT00000362751.1,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000210195.2,chrM,ENSEMBL,transcript,15888,15953,NaN,+,0,ENST00000387460.2,Mt_tRNA,...,J01415.22,Mt_tRNA,NOVEL,J01415.22-201,3,,,,,
ENSG00000210195.2,chrM,ENSEMBL,exon,15888,15953,NaN,+,0,ENST00000387460.2,Mt_tRNA,...,J01415.22,Mt_tRNA,NOVEL,J01415.22-201,3,,,,,
ENSG00000210196.2,chrM,ENSEMBL,gene,15956,16023,NaN,-,0,ENSG00000210196.2,Mt_tRNA,...,J01415.23,Mt_tRNA,NOVEL,J01415.23,3,,,,,


In [11]:
# Select rows with our gene symbol
my_gene_symbol = anno.loc[my_gene["Gene_Symbol"]]
my_gene_symbol

,seqname,source,feature,start,end,score,strand,frame,transcript_id,gene_type,...,gene_name,transcript_type,transcript_status,transcript_name,level,havana_gene,havana_transcript,ont,tag,ccdsid
gene_id,,,,,,,,,,,,,,,,,,,,,
ENSG00000226958.1,chrX,HAVANA,gene,108297361,108297792,NaN,-,0,ENSG00000226958.1,pseudogene,...,RN28S1,pseudogene,KNOWN,RN28S1,2,OTTHUMG00000040033.1,,,,
ENSG00000226958.1,chrX,HAVANA,transcript,108297361,108297792,NaN,-,0,ENST00000419932.1,pseudogene,...,RN28S1,processed_pseudogene,KNOWN,RN28S1-001,2,OTTHUMG00000040033.1,OTTHUMT00000096568.1,PGO:0000004,,
ENSG00000226958.1,chrX,HAVANA,exon,108297361,108297792,NaN,-,0,ENST00000419932.1,pseudogene,...,RN28S1,processed_pseudogene,KNOWN,RN28S1-001,2,OTTHUMG00000040033.1,OTTHUMT00000096568.1,PGO:0000004,,


In [12]:
print(my_gene_symbol['gene_name'])

gene_id
ENSG00000226958.1    RN28S1
ENSG00000226958.1    RN28S1
ENSG00000226958.1    RN28S1
Name: gene_name, dtype: object


In [27]:
# 2. Next, randomly pick a sample from the 660 listed samples to work with.
sample = rna['HG00097.7.M_120219_2']     # I chose the second sample
sample

0         0.00000
1         0.00000
2         0.00000
3         0.66697
4         0.00000
           ...   
183081    0.25167
183082    0.00000
183083    0.13191
183084    0.18754
183085    0.07601
Name: HG00097.7.M_120219_2, Length: 183086, dtype: float64

In [ ]:
# 2.a Make a histogram of the RPKM counts for your sample in the Geuvadis data.
_ = plt.hist(sample, bins='auto')
_ = plt.ylabel('Counts')

In [ ]:
# 2.b Convert RPKM values to TPM, and make a histogram of the TPM values
    # rpkm = gene_reads / (gene_length_in_kb * total_reads)
    # tpm = (rpk / rpk_sum) 

In [ ]:
# 2.c Make a scatter plot of TPM vs RPKM values